In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 71.0 MB/s eta 0:00:00


In [ ]:
import gensim
from gensim.models import Word2Vec, FastText
import torch
import torch.nn as nn
import numpy as np

# ==========================================
# 1. Текстовый корпус банковских обращений
# ==========================================
# Токенизированные предложения (списки слов)
corpus = [
    ["заблокировали", "дебетовую", "карту", "без", "предупреждения"],
    ["не", "могу", "перевести", "деньги", "через", "приложение"],
    ["приложение", "вылетает", "при", "входе", "в", "личный", "кабинет"],
    ["подозрительное", "списание", "средств", "со", "счета", "мошенники"],
    ["карта", "заблокирована", "банком", "как", "разблокировать"],
    ["быстрый", "и", "удобный", "перевод", "денег", "на", "кредитную", "карту"],
    ["ужасный", "сервис", "поддержка", "не", "отвечает", "целый", "день"],
    ["украли", "деньги", "с", "кредитной", "карты", "списали", "все", "средства"],
    ["не", "работает", "перевод", "по", "номеру", "телефона"],
    ["требую", "вернуть", "деньги", "заблокировали", "счет", "мошенничество"]
] * 30  # Умножаем корпус для стабильности обучения

In [ ]:
# ==========================================
# ЗАДАНИЕ 1: Обучение Word2Vec и FastText в Gensim
# ==========================================
# TODO 1:
# 1. Обучите модель Word2Vec (vector_size=32, window=3, min_count=1, sg=1 (Skip-gram), epochs=20)
# 2. Обучите модель FastText с точно такими же параметрами

w2v_model = Word2Vec(sentences=corpus, vector_size=32, window=3, min_count=1, sg=1, epochs=20)
ft_model = FastText(sentences=corpus, vector_size=32, window=3, min_count=1, sg=1, epochs=20)

In [ ]:
# ==========================================
# ЗАДАНИЕ 2: Проверка OOV (Out-Of-Vocabulary) и синонимов
# ==========================================
# TODO 2:
# 1. Найдите топ-3 похожих слова к слову "карта" в Word2Vec через .wv.most_similar()
print(w2v_model.wv.most_similar('карта', topn=3), '\n')

# 2. Попробуйте проверить слово с опечаткой: "каарту" или "заблоокировали"
#    - Что произойдет при вызове w2v_model.wv["каарту"]? (подсказка: обработайте KeyError через try/except)
try:
  vec = w2v_model.wv["каарту"]
  print(vec)
except KeyError:
  print("Слова нет в словаре w2v\n")

#    - Что произойдет при вызове ft_model.wv["каарту"]?
ft_vec = ft_model.wv["каарту"]
print(ft_vec, '\n')

# 3. Выведите косинусное сходство между "дебетовую" и "кредитную" в обеих моделях (.wv.similarity)
print("Word2Vec", w2v_model.wv.similarity(w1="дебетовую", w2="кредитную"))

[('заблокирована', 0.9895570874214172), ('как', 0.9881019592285156), ('банком', 0.9870731830596924)] 

Слова нет в словаре w2v

[ 0.06742418 -0.05377142  0.1466709   0.07277422  0.02089785 -0.00922641
 -0.10672273 -0.13078627  0.04839511 -0.02890389  0.02841568 -0.03971962
 -0.02184662 -0.03543369 -0.00823298  0.01664742 -0.02593401  0.05671477
  0.10411137  0.05266158 -0.13082491  0.14679573  0.10859856 -0.08426955
  0.04044362  0.25709546 -0.21945617  0.0439407  -0.07694025 -0.1404022
  0.02279696  0.11589578] 

Word2Vec 0.9634765


In [ ]:
# ==========================================
# ЗАДАНИЕ 3: Загрузка матрицы весов FastText в PyTorch nn.Embedding
# ==========================================
# TODO 3:
# 1. Создайте словарь id2word (список всех слов из ft_model.wv.index_to_key)
id2word = ft_model.wv.index_to_key
print(id2word[0:5])

# 2. Добавьте в начало словаря спец-токены: "<pad>" (индекс 0) и "<unk>" (индекс 1)
id2word = ['<pad>', '<unk>'] + list(id2word)
word2id = {w: i for i, w in enumerate(id2word)}

# 3. Сформируйте numpy-матрицу весов weights_matrix формы [vocab_size, vector_size=32]:
#    - Для <pad> заполните строку нулями
#    - Для <unk> заполните случайными числами или средним вектором
#    - Для остальных слов скопируйте их векторы из ft_model.wv[word]
vocab_size = len(id2word)
vector_size = ft_model.wv.vector_size

weights_matrix = np.zeros((vocab_size, vector_size), dtype=np.float32)
weights_matrix[1] = ft_model.wv.vectors.mean(axis=0)

for i, word in enumerate(id2word[2:], start=2):
  weights_matrix[i] = ft_model.wv[word]
print(weights_matrix.shape)

# 4. Инициализируйте PyTorch слой nn.Embedding.from_pretrained(torch.FloatTensor(weights_matrix), freeze=True)
embedding = nn.Embedding.from_pretrained(
    torch.FloatTensor(weights_matrix),
    freeze=True,
    padding_idx=0
)
print(embedding.weight.shape)
print(embedding.weight.requires_grad) # потому что заморозили градиенты для обучения

# 5. Передайте тензор токенов tensor([[2, 3, 0]]) через ваш embedding слой и выведите полученную форму (shape).
tokens = torch.tensor([[2, 3, 0]])
print("tokens.shape", tokens.shape)

embed = embedding(tokens)
print(embed.shape)


['деньги', 'не', 'перевод', 'приложение', 'карту']
(58, 32)
torch.Size([58, 32])
False
tokens.shape torch.Size([1, 3])
torch.Size([1, 3, 32])
